## ANN Tutorial 
A tutorial for a Artificial Neural Network touching all most significant milestones.

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import log_loss, accuracy_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)

In [2]:
# Creation of a synthetic dataset for credit risk modeling

# Per rendere l'esperimento riproducibile
np.random.seed(42)

# Numero di imprese simulate
n = 2000

# -----------------------------
# 1. Creazione delle features
# -----------------------------

debt_to_assets = np.random.uniform(0.10, 0.95, n)

interest_coverage = np.random.gamma(
    shape=2.0,
    scale=2.0,
    size=n
)

roa = np.random.normal(
    loc=0.05,
    scale=0.08,
    size=n
)

current_ratio = np.random.normal(
    loc=1.5,
    scale=0.5,
    size=n
)

revenue_growth = np.random.normal(
    loc=0.03,
    scale=0.12,
    size=n
)

# Evitiamo valori economicamente assurdi
interest_coverage = np.clip(interest_coverage, 0, 15)
current_ratio = np.clip(current_ratio, 0.2, 4)

# -----------------------------
# 2. Generazione della PD "vera"
# -----------------------------

# Funzione latente: NON sarà fornita alla ANN
z = (
    -2.0
    + 4.0 * debt_to_assets
    - 0.35 * interest_coverage
    - 5.0 * roa
    - 0.8 * current_ratio
    - 2.0 * revenue_growth
)

# Logistic function:
# trasforma z in una probabilità compresa tra 0 e 1
true_pd = 1 / (1 + np.exp(-z))

# -----------------------------
# 3. Generazione del default
# -----------------------------

# Ogni impresa può defaultare oppure no
default = np.random.binomial(
    n=1,
    p=true_pd
)

# -----------------------------
# 4. DataFrame finale
# -----------------------------

df = pd.DataFrame({
    "Debt_to_Assets": debt_to_assets,
    "Interest_Coverage": interest_coverage,
    "ROA": roa,
    "Current_Ratio": current_ratio,
    "Revenue_Growth": revenue_growth,
    "Default": default
})

print(df.head())

print("\nDataset shape:", df.shape)

print("\nDefault rate:")
print(df["Default"].mean())

   Debt_to_Assets  Interest_Coverage       ROA  Current_Ratio  Revenue_Growth  \
0        0.418359           1.541461 -0.017715       2.387745       -0.268495   
1        0.908107           1.621719  0.124054       0.760402       -0.024987   
2        0.722195           6.292374  0.023538       1.569826        0.229703   
3        0.608860           1.659870  0.009187       1.532512       -0.027128   
4        0.232616           3.278467  0.061956       1.906640       -0.028873   

   Default  
0        0  
1        1  
2        0  
3        0  
4        0  

Dataset shape: (2000, 6)

Default rate:
0.114


In [4]:
# ============================================================
# 1. SEPARAZIONE FEATURES (X) E TARGET (y)
# ============================================================

X = df.drop(columns=["Default"])
y = df["Default"]

print("Shape X:", X.shape)
print("Shape y:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nPrime 5 osservazioni di X:")
print(X.head())

print("\nPrime 5 osservazioni di y:")
print(y.head())

Shape X: (2000, 5)
Shape y: (2000,)

Features:
['Debt_to_Assets', 'Interest_Coverage', 'ROA', 'Current_Ratio', 'Revenue_Growth']

Prime 5 osservazioni di X:
   Debt_to_Assets  Interest_Coverage       ROA  Current_Ratio  Revenue_Growth
0        0.418359           1.541461 -0.017715       2.387745       -0.268495
1        0.908107           1.621719  0.124054       0.760402       -0.024987
2        0.722195           6.292374  0.023538       1.569826        0.229703
3        0.608860           1.659870  0.009187       1.532512       -0.027128
4        0.232616           3.278467  0.061956       1.906640       -0.028873

Prime 5 osservazioni di y:
0    0
1    1
2    0
3    0
4    0
Name: Default, dtype: int32


In [5]:
# ============================================================
# 2. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# Primo split:
# 70% training
# 30% temporaneamente lasciato fuori

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Secondo split:
# il 30% residuo viene diviso a metà:
# 15% validation
# 15% test

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("TRAIN")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nVALIDATION")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nTEST")
print("X:", X_test.shape)
print("y:", y_test.shape)

TRAIN
X: (1400, 5)
y: (1400,)

VALIDATION
X: (300, 5)
y: (300,)

TEST
X: (300, 5)
y: (300,)


In [6]:
# Check of core metrics: default rate in each dataset

print("Default rate - Entire dataset:",
      round(y.mean(), 4))

print("Default rate - Training:",
      round(y_train.mean(), 4))

print("Default rate - Validation:",
      round(y_val.mean(), 4))

print("Default rate - Test:",
      round(y_test.mean(), 4))

print(X_train.describe().T)

Default rate - Entire dataset: 0.114
Default rate - Training: 0.1143
Default rate - Validation: 0.1133
Default rate - Test: 0.1133
                    count      mean       std       min       25%       50%  \
Debt_to_Assets     1400.0  0.526338  0.247001  0.102736  0.307283  0.531788   
Interest_Coverage  1400.0  4.049271  2.783573  0.060596  1.990436  3.452853   
ROA                1400.0  0.045424  0.080791 -0.181961 -0.007692  0.045314   
Current_Ratio      1400.0  1.515846  0.494859  0.200000  1.178954  1.497112   
Revenue_Growth     1400.0  0.025851  0.117624 -0.440688 -0.053816  0.025762   

                        75%        max  
Debt_to_Assets     0.743180   0.948595  
Interest_Coverage  5.394425  15.000000  
ROA                0.098376   0.332324  
Current_Ratio      1.854505   3.188691  
Revenue_Growth     0.106519   0.362205  


In [ ]:
# ============================================================
# STANDARDIZATION
# ============================================================

scaler = StandardScaler()

# IMPORTANTE:
# fit SOLO sul training set
scaler.fit(X_train)

# Applichiamo poi la stessa trasformazione ai tre dataset
X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Original training data:")
print(X_train.head())

print("\nStandardized training data:")
print(X_train_scaled[:5])

scaling_parameters = pd.DataFrame({
    "Feature": X.columns,
    "Mean": scaler.mean_,
    "Std": scaler.scale_
})

print(scaling_parameters)

Original training data:
      Debt_to_Assets  Interest_Coverage       ROA  Current_Ratio  \
520         0.605011           1.989267  0.094475       1.350242   
256         0.895732           3.830921  0.128083       0.972005   
1190        0.157248           3.028011  0.117401       1.205373   
1644        0.175756           4.352433  0.199686       1.464888   
584         0.638391           4.658916  0.086586       1.553992   

      Revenue_Growth  
520         0.101383  
256         0.123336  
1190        0.247387  
1644       -0.069518  
584        -0.103170  

Standardized training data:
[[ 0.31862739 -0.7403222   0.60734961 -0.33476765  0.64236698]
 [ 1.49605113 -0.07847047  1.02348252 -1.09937449  0.82907176]
 [-1.49482228 -0.3670195   0.89121639 -0.62762102  1.88408514]
 [-1.41986549  0.1089498   1.91006582 -0.10301136 -0.81109042]
 [ 0.45381552  0.2190935   0.50966354  0.07711191 -1.09728613]]
             Feature      Mean       Std
0     Debt_to_Assets  0.526338  0.246912
1 

In [11]:
print("Mean after standardization:")
print(X_train_scaled.mean(axis=0))

print("\nStandard deviation after standardization:")
print(X_train_scaled.std(axis=0))

print("TRAIN")
print("Mean:", X_train_scaled.mean(axis=0))
print("Std: ", X_train_scaled.std(axis=0))

print("\nVALIDATION")
print("Mean:", X_val_scaled.mean(axis=0))
print("Std: ", X_val_scaled.std(axis=0))

print("\nTEST")
print("Mean:", X_test_scaled.mean(axis=0))
print("Std: ", X_test_scaled.std(axis=0))

Mean after standardization:
[-2.28388736e-17  6.85166209e-17 -1.01506105e-17  1.39570895e-16
 -5.07530526e-18]

Standard deviation after standardization:
[1. 1. 1. 1. 1.]
TRAIN
Mean: [-2.28388736e-17  6.85166209e-17 -1.01506105e-17  1.39570895e-16
 -5.07530526e-18]
Std:  [1. 1. 1. 1. 1.]

VALIDATION
Mean: [ 0.02900029 -0.02662184  0.04708877 -0.016693   -0.02394753]
Std:  [0.99943027 0.92800795 1.01387798 1.10936948 0.98299195]

TEST
Mean: [-0.09641111 -0.06306022  0.08279493 -0.00233644  0.01812851]
Std:  [1.03332312 0.95760634 1.06999876 1.00983206 0.95751535]


In [16]:
# Implement the ML Model

warnings.filterwarnings("ignore", category=ConvergenceWarning)

model = MLPClassifier(
    hidden_layer_sizes=(4,),   # una hidden layer con 4 neuroni
    activation="relu",        # activation function hidden layer
    solver="adam",            # algoritmo di ottimizzazione
    learning_rate_init=0.001,
    alpha=0.0,                # per ora NO L2 regularization
    max_iter=1,               # per ora solo 1 epoch
    warm_start=True,          # permette di continuare il training
    random_state=42
)

print(model)

model.fit(X_train_scaled, y_train)

MLPClassifier(alpha=0.0, hidden_layer_sizes=(4,), max_iter=1, random_state=42,
              warm_start=True)


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(4,)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previouscall to fit as initialization, otherwise, just erase theprevious solution. See :term:`the Glossary <warm_start>`.",True
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5


In [19]:
# Explore the model

print("Number of layers:", model.n_layers_)
print("Number of outputs:", model.n_outputs_)

print("\nWeight matrices:")
for i, weights in enumerate(model.coefs_):
    print(f"Layer {i}: {weights.shape}")

print("\nBias vectors:")
for i, bias in enumerate(model.intercepts_):
    print(f"Layer {i}: {bias.shape}")

print("\nINPUT → HIDDEN weights:")
print(model.coefs_[0])

print("\nHIDDEN biases:")
print(model.intercepts_[0])

print("\nHIDDEN → OUTPUT weights:")
print(model.coefs_[1])

print("\nOUTPUT bias:")
print(model.intercepts_[1])

Number of layers: 3
Number of outputs: 1

Weight matrices:
Layer 0: (5, 4)
Layer 1: (4, 1)

Bias vectors:
Layer 0: (4,)
Layer 1: (1,)

INPUT → HIDDEN weights:
[[-0.211589    0.72934871  0.38001424  0.15615312]
 [-0.56862134 -0.55624088 -0.72717205  0.59119193]
 [ 0.17184112  0.3329332  -0.78878886  0.7604442 ]
 [ 0.54983074 -0.46403435 -0.52554939 -0.51039972]
 [-0.32560033  0.03696426 -0.11015317 -0.33465658]]

HIDDEN biases:
[ 0.18967662 -0.59552022 -0.33260347 -0.22510412]

HIDDEN → OUTPUT weights:
[[-0.10325951]
 [ 0.61820369]
 [-0.66353066]
 [ 0.02424438]]

OUTPUT bias:
[0.19547595]


In [20]:
# Test a prediction on test set 

company = X_test_scaled[0].reshape(1, -1)

pd_hat = model.predict_proba(company)[0, 1]
prediction = model.predict(company)[0]

print("Estimated PD:", pd_hat)
print("Predicted class:", prediction)
print("Actual class:", y_test.iloc[0])

Estimated PD: 0.5144416478666162
Predicted class: 1
Actual class: 0


In [48]:
# Manually train the model for multiple epochs and track loss and accuracy

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

epochs = 150

for epoch in range(epochs):

    # Un'altra epoch di training
    model.fit(X_train_scaled, y_train)

    # Probabilità predette
    train_prob = model.predict_proba(X_train_scaled)[:, 1]
    val_prob = model.predict_proba(X_val_scaled)[:, 1]

    # Classi predette
    train_pred = model.predict(X_train_scaled)
    val_pred = model.predict(X_val_scaled)

    # Loss
    train_loss = log_loss(y_train, train_prob)
    val_loss = log_loss(y_val, val_prob)

    # Accuracy
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    # Stampiamo qualche checkpoint
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

Epoch  10 | Train Loss: 0.2818 | Val Loss: 0.3209 | Train Acc: 0.8843 | Val Acc: 0.8800
Epoch  20 | Train Loss: 0.2818 | Val Loss: 0.3210 | Train Acc: 0.8843 | Val Acc: 0.8800
Epoch  30 | Train Loss: 0.2819 | Val Loss: 0.3210 | Train Acc: 0.8843 | Val Acc: 0.8800
Epoch  40 | Train Loss: 0.2819 | Val Loss: 0.3210 | Train Acc: 0.8836 | Val Acc: 0.8800
Epoch  50 | Train Loss: 0.2819 | Val Loss: 0.3208 | Train Acc: 0.8836 | Val Acc: 0.8767
Epoch  60 | Train Loss: 0.2818 | Val Loss: 0.3207 | Train Acc: 0.8836 | Val Acc: 0.8767
Epoch  70 | Train Loss: 0.2818 | Val Loss: 0.3206 | Train Acc: 0.8836 | Val Acc: 0.8767
Epoch  80 | Train Loss: 0.2818 | Val Loss: 0.3205 | Train Acc: 0.8836 | Val Acc: 0.8767
Epoch  90 | Train Loss: 0.2818 | Val Loss: 0.3204 | Train Acc: 0.8843 | Val Acc: 0.8767
Epoch 100 | Train Loss: 0.2818 | Val Loss: 0.3203 | Train Acc: 0.8850 | Val Acc: 0.8767
Epoch 110 | Train Loss: 0.2818 | Val Loss: 0.3202 | Train Acc: 0.8850 | Val Acc: 0.8767
Epoch 120 | Train Loss: 0.2818 |

In [49]:
# Explore model output

best_epoch = np.argmin(val_losses) + 1
best_val_loss = np.min(val_losses)

print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))
print("Final training loss:", round(train_losses[-1], 4))
print("Final validation loss:", round(val_losses[-1], 4))

final_gap = val_losses[-1] - train_losses[-1]

print("Final train-validation loss gap:", round(final_gap, 4))

# Check the model on the test set

test_prob = model.predict_proba(X_test_scaled)[:, 1]
test_pred = model.predict(X_test_scaled)

results = X_test.copy()

results["Estimated_PD"] = test_prob
results["Predicted_Default"] = test_pred
results["Actual_Default"] = y_test.values

print(
    results[
        [
            "Debt_to_Assets",
            "Interest_Coverage",
            "ROA",
            "Current_Ratio",
            "Revenue_Growth",
            "Estimated_PD",
            "Predicted_Default",
            "Actual_Default"
        ]
    ].head(15)
)

Best epoch: 149
Best validation loss: 0.3199
Final training loss: 0.2818
Final validation loss: 0.3199
Final train-validation loss gap: 0.0382
      Debt_to_Assets  Interest_Coverage       ROA  Current_Ratio  \
358         0.899196           1.034966  0.056046       2.156891   
1842        0.473494           1.009706  0.111311       1.608345   
1081        0.175999           5.411637  0.054539       2.067810   
1928        0.645977           1.121924  0.051846       1.785473   
1078        0.285516           1.289027  0.122817       2.729936   
386         0.339729           3.873854  0.143401       1.831268   
291         0.131746           5.211781  0.019243       1.904286   
1214        0.689630           1.731396 -0.094300       0.971082   
589         0.229117           5.479529  0.205380       1.181096   
1062        0.753032           3.467281  0.163313       1.240681   
590         0.218003           4.159223 -0.089778       1.471082   
1795        0.172073           6.117295  

In [50]:
# Check edge cases

results["Distance_from_05"] = abs(results["Estimated_PD"] - 0.5)

uncertain_cases = results.sort_values(
    "Distance_from_05"
).head(10)

print(
    uncertain_cases[
        [
            "Estimated_PD",
            "Predicted_Default",
            "Actual_Default"
        ]
    ]
)

      Estimated_PD  Predicted_Default  Actual_Default
707       0.506555                  1               1
1772      0.492490                  0               0
1088      0.511409                  1               0
1333      0.514351                  1               1
1517      0.481035                  0               0
1838      0.521182                  1               1
150       0.469897                  0               0
540       0.457526                  0               1
1681      0.552760                  1               0
996       0.446338                  0               0


In [51]:
# Check performance metrics on the test set

test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred)
test_recall = recall_score(y_test, test_pred)
test_auc = roc_auc_score(y_test, test_prob)

print("TEST PERFORMANCE")
print("----------------")
print("Accuracy :", round(test_accuracy, 4))
print("Precision:", round(test_precision, 4))
print("Recall   :", round(test_recall, 4))
print("ROC AUC  :", round(test_auc, 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_pred))

TEST PERFORMANCE
----------------
Accuracy : 0.89
Precision: 0.5556
Recall   : 0.1471
ROC AUC  : 0.8578

Confusion matrix:
[[262   4]
 [ 29   5]]


In [52]:
# Since performance metrics can be sensitive to the threshold, let's explore how changing the threshold affects precision and recall.

threshold = 0.20

test_pred_20 = (test_prob >= threshold).astype(int)

print("Threshold:", threshold)
print("Accuracy :", round(accuracy_score(y_test, test_pred_20), 4))
print("Precision:", round(precision_score(y_test, test_pred_20), 4))
print("Recall   :", round(recall_score(y_test, test_pred_20), 4))

cm = confusion_matrix(y_test, test_pred_20)

cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Default", "Actual: Default"],
    columns=["Predicted: No Default", "Predicted: Default"]
)

print(cm_df)

Threshold: 0.2
Accuracy : 0.8567
Precision: 0.4151
Recall   : 0.6471
                    Predicted: No Default  Predicted: Default
Actual: No Default                    235                  31
Actual: Default                        12                  22


In [53]:
# ============================================================
# ANN vs LOGISTIC REGRESSION
# ============================================================
#
# OBIETTIVO:
# Confrontiamo la nostra Artificial Neural Network con un modello
# molto più semplice: la Logistic Regression.
#
# Il confronto è particolarmente interessante perché sappiamo che
# il dataset sintetico è stato generato utilizzando una relazione
# logistica:
#
#       PD = sigmoid(beta0 + beta1*x1 + ... + beta5*x5)
#
# Quindi la Logistic Regression è, per costruzione, molto vicina
# alla vera struttura sottostante ai dati.
#
# La ANN, invece, possiede maggiore flessibilità grazie alla hidden
# layer, ma questa maggiore complessità NON implica necessariamente
# una migliore capacità predittiva.
#
# Per rendere il confronto corretto utilizziamo:
# - stesso training set
# - stesso test set
# - stesse features
# - stessa standardizzazione
# - stesse metriche
#
# Confronteremo soprattutto ROC-AUC, perché misura la capacità del
# modello di discriminare/rankare default e non-default senza
# dipendere da una singola classification threshold.
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. TRAINING DELLA LOGISTIC REGRESSION
# ------------------------------------------------------------

logit = LogisticRegression(
    random_state=42
)

logit.fit(X_train_scaled, y_train)


# ------------------------------------------------------------
# 2. PREDICTION SUL TEST SET
# ------------------------------------------------------------

# Probabilità di default stimate
logit_prob = logit.predict_proba(X_test_scaled)[:, 1]

# Classificazione standard con threshold = 0.50
logit_pred = logit.predict(X_test_scaled)


# ------------------------------------------------------------
# 3. PERFORMANCE LOGISTIC REGRESSION
# ------------------------------------------------------------

logit_accuracy = accuracy_score(y_test, logit_pred)
logit_precision = precision_score(y_test, logit_pred)
logit_recall = recall_score(y_test, logit_pred)
logit_auc = roc_auc_score(y_test, logit_prob)

print("LOGISTIC REGRESSION - TEST PERFORMANCE")
print("--------------------------------------")
print("Accuracy :", round(logit_accuracy, 4))
print("Precision:", round(logit_precision, 4))
print("Recall   :", round(logit_recall, 4))
print("ROC AUC  :", round(logit_auc, 4))


# ------------------------------------------------------------
# 4. CONFUSION MATRIX
# ------------------------------------------------------------

cm_logit = confusion_matrix(y_test, logit_pred)

cm_logit_df = pd.DataFrame(
    cm_logit,
    index=["Actual: No Default", "Actual: Default"],
    columns=["Predicted: No Default", "Predicted: Default"]
)

print("\nConfusion Matrix:")
print(cm_logit_df)


# ------------------------------------------------------------
# 5. CONFRONTO ANN vs LOGISTIC REGRESSION
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "ANN": [
        accuracy_score(y_test, test_pred),
        precision_score(y_test, test_pred),
        recall_score(y_test, test_pred),
        roc_auc_score(y_test, test_prob)
    ],
    
    "Logistic Regression": [
        logit_accuracy,
        logit_precision,
        logit_recall,
        logit_auc
    ]
},
    index=["Accuracy", "Precision", "Recall", "ROC AUC"]
)

print("\nMODEL COMPARISON")
print("----------------")
print(comparison.round(4))

LOGISTIC REGRESSION - TEST PERFORMANCE
--------------------------------------
Accuracy : 0.88
Precision: 0.3333
Recall   : 0.0588
ROC AUC  : 0.8626

Confusion Matrix:
                    Predicted: No Default  Predicted: Default
Actual: No Default                    262                   4
Actual: Default                        32                   2

MODEL COMPARISON
----------------
              ANN  Logistic Regression
Accuracy   0.8900               0.8800
Precision  0.5556               0.3333
Recall     0.1471               0.0588
ROC AUC    0.8578               0.8626


In [56]:
# ============================================================
# LOGISTIC REGRESSION WITH 20% THRESHOLD
#
# La Logistic Regression produce una PD continua.
# Qui NON riaddestriamo il modello: cambiamo solamente la
# decision threshold da 50% a 20%, esattamente come abbiamo
# fatto precedentemente con la ANN.
# ============================================================

threshold = 0.20

logit_pred_20 = (logit_prob >= threshold).astype(int)

print("LOGISTIC REGRESSION - THRESHOLD 20%")
print("-----------------------------------")

print("Accuracy :",
      round(accuracy_score(y_test, logit_pred_20), 4))

print("Precision:",
      round(precision_score(y_test, logit_pred_20), 4))

print("Recall   :",
      round(recall_score(y_test, logit_pred_20), 4))

print("ROC AUC  :",
      round(roc_auc_score(y_test, logit_prob), 4))


cm = confusion_matrix(y_test, logit_pred_20)

cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Default", "Actual: Default"],
    columns=["Predicted: No Default", "Predicted: Default"]
)

print("\nConfusion Matrix:")
print(cm_df)

print("\nLogistic Regression Coefficients:")
print(logit.coef_)

LOGISTIC REGRESSION - THRESHOLD 20%
-----------------------------------
Accuracy : 0.8667
Precision: 0.4423
Recall   : 0.6765
ROC AUC  : 0.8626

Confusion Matrix:
                    Predicted: No Default  Predicted: Default
Actual: No Default                    237                  29
Actual: Default                        11                  23

Logistic Regression Coefficients:
[[ 0.9966296  -0.87970895 -0.44938688 -0.40390597 -0.2127128 ]]


In [57]:
# ============================================================
# BACK-TRANSFORMATION OF LOGISTIC REGRESSION COEFFICIENTS
#
# Obiettivo:
# La Logistic Regression è stata stimata usando features
# standardizzate. Riportiamo ora coefficienti e intercept
# sulla scala originale dei dati, per poterli confrontare
# direttamente con i coefficienti "veri" usati per generare
# il dataset sintetico.
# ============================================================

# Coefficienti stimati sulla scala standardizzata
beta_scaled = logit.coef_[0]
intercept_scaled = logit.intercept_[0]

# Parametri dello StandardScaler calcolati sul training set
mu = scaler.mean_
sigma = scaler.scale_

# ------------------------------------------------------------
# 1. Coefficienti sulla scala originale
# ------------------------------------------------------------

beta_original = beta_scaled / sigma

# ------------------------------------------------------------
# 2. Intercept sulla scala originale
# ------------------------------------------------------------

intercept_original = (
    intercept_scaled
    - np.sum(beta_scaled * mu / sigma)
)

# ------------------------------------------------------------
# 3. Coefficienti veri usati per generare il dataset
# ------------------------------------------------------------

beta_true = np.array([
    4.0,     # Debt_to_Assets
   -0.35,    # Interest_Coverage
   -5.0,     # ROA
   -0.8,     # Current_Ratio
   -2.0      # Revenue_Growth
])

intercept_true = -2.0

# ------------------------------------------------------------
# 4. Confronto
# ------------------------------------------------------------

comparison_coef = pd.DataFrame({
    "Feature": X.columns,
    "True coefficient": beta_true,
    "Estimated coefficient": beta_original,
    "Difference": beta_original - beta_true
})

print(comparison_coef.round(4))

print("\nIntercept")
print("True intercept     :", round(intercept_true, 4))
print("Estimated intercept:", round(intercept_original, 4))

             Feature  True coefficient  Estimated coefficient  Difference
0     Debt_to_Assets              4.00                 4.0364      0.0364
1  Interest_Coverage             -0.35                -0.3161      0.0339
2                ROA             -5.00                -5.5643     -0.5643
3      Current_Ratio             -0.80                -0.8165     -0.0165
4     Revenue_Growth             -2.00                -1.8091      0.1909

Intercept
True intercept     : -2.0
Estimated intercept: -2.0409
